In [2]:
!pip install transformers==4.36.2 accelerate==0.27.2 -i https://mirrors.aliyun.com/pypi/simple/

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 1.2 MB/s eta 0:00:0000:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 49.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 1.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 1.3 MB/s eta 0:00:00a 0:00:010m
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing insta

In [1]:
import torch
import warnings
warnings.filterwarnings("ignore") # 忽略警告
from transformers import AutoTokenizer, AutoModel, AutoConfig

model_path = "/mnt/workspace/chatglm3-6b"

print("1. 正在加载 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

print("2. 正在修复 Config 配置...")
# 修复官方 config.json 漏掉的参数
config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
config.max_length = 8192

print("3. 正在全速加载模型 (不卡顿，约 1 分钟)...")
# low_cpu_mem_usage=False 避开死锁 Bug
model = AutoModel.from_pretrained(
    model_path, 
    config=config,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=False  
)

# 直接转 float() 在 CPU 跑得最快
model = model.float().eval()

print("\n✅ 加载成功！开始极速回答作业问题：\n")
print("="*60)
# 📝 自动化测试题目
questions = [
    "请说出以下两句话区别在哪里？ 1、冬天：能穿多少穿多少 2、夏天：能穿多少穿多少",
    "请说出以下两句话区别在哪里？单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上",
    "他知道我知道你知道他不知道吗？ 这句话里，到底谁不知道",
    "明明明明明白白白喜欢他，可她就是不说。 这句话里，明明和白白谁喜欢谁？",
    "领导：你这是什么意思？ 小明：没什么意思。意思意思。 领导：你这就不够意思了。 小明：小意思，小意思。领导：你这人真有意思。 小明：其实也没有别的意思。 领导：那我就不好意思了。 小明：是我不好意思。请问：以上“意思”分别是什么意思。"

]

for q in questions:
    print(f"❓ 问：{q}")
    # 量化后的模型在 CPU 上出字速度会比 FP16 快一些
    response, _ = model.chat(tokenizer, q, history=[])
    print(f"💡 答：\n{response}")
    print("-" * 60)

1. 正在加载 Tokenizer...


Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


2. 正在修复 Config 配置...
3. 正在全速加载模型 (不卡顿，约 1 分钟)...


Loading checkpoint shards: 100%|██████████| 7/7 [00:44<00:00,  6.30s/it]



✅ 加载成功！开始极速回答作业问题：

❓ 问：请说出以下两句话区别在哪里？ 1、冬天：能穿多少穿多少 2、夏天：能穿多少穿多少
💡 答：
这两句话的区别在于所处的季节不同。

1. 冬天：能穿多少穿多少，表示在寒冷的冬季，人们为了保暖，会尽量多穿一些衣服， even if it makes them feel hot and uncomfortable.

2. 夏天：能穿多少穿多少，表示在炎热的夏季，人们为了散热，会尽量少穿一些衣服， even if it makes them feel cool and comfortable.
------------------------------------------------------------
❓ 问：请说出以下两句话区别在哪里？单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上
💡 答：
这两句话的 differ 在于重复使用了一个相同的短语,但前一句话使用了两个逗号来分隔原因,后一句话则没有使用逗号。因此,两句话在语法和可读性上有所不同。
------------------------------------------------------------
❓ 问：他知道我知道你知道他不知道吗？ 这句话里，到底谁不知道
💡 答：
这句话是一个典型的谜语或绕口令,通常被称为“不知道你知道我知道他不知道”或“我知道你知道我知道你知道他不知道”。

在这个绕口令中,每个人都在说他们知道的事情,但是没有人知道的事情是“我不知道你知道我知道他不知道”。因此,最终的问题是谁不知道,答案是“我不知道你知道我知道他不知道”。
------------------------------------------------------------
❓ 问：明明明明明白白白喜欢他，可她就是不说。 这句话里，明明和白白谁喜欢谁？
💡 答：
根据这句话的描述,明明明白白白喜欢他,但白白并没有表达出自己的喜欢。因此,可以理解为明明喜欢白白,但白白并没有表达出来。
------------------------------------------------------------
❓ 问：领导：你这是什么意思？ 小明：没什么意思。意思意思。 领导：你这就不够意思了。 小明：小意思，小意思。领导：你这人真有意思。 小明

In [1]:
import torch
import warnings
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedModel

# 🚀 补丁：防止新旧版本 transformers 冲突
if not hasattr(PreTrainedModel, "all_tied_weights_keys"):
    PreTrainedModel.all_tied_weights_keys = property(lambda self: {})

model_path = "/mnt/workspace/Qwen-7B-Chat"

print("1. 正在加载 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

print("2. 正在加载 Qwen-7B 模型 (14GB体积，绝对不会爆内存)...")
# 🔑 极速且省内存的秘诀：
# 使用 torch.bfloat16，模型只占 14GB，且 CPU 原生支持，不需要转 float()
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,  # <--- 关键修改：用 bfloat16
    low_cpu_mem_usage=False  
).eval()  # <--- 删掉了导致爆内存的 .float()

print("\n✅ Qwen-7B-Chat 加载成功！内存安全，开始极速回答作业问题：\n")
print("="*60)

# 📝 自动化测试题目
questions =[
    "请说出以下两句话区别在哪里？ 1、冬天：能穿多少穿多少 2、夏天：能穿多少穿多少",
    "请说出以下两句话区别在哪里？单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上",
    "他知道我知道你知道他不知道吗？ 这句话里，到底谁不知道",
    "明明明明明白白白喜欢他，可她就是不说。 这句话里，明明和白白谁喜欢谁？",
    "领导：你这是什么意思？ 小明：没什么意思。意思意思。 领导：你这就不够意思了。 小明：小意思，小意思。领导：你这人真有意思。 小明：其实也没有别的意思。 领导：那我就不好意思了。 小明：是我不好意思。请问：以上“意思”分别是什么意思。"
]

for q in questions:
    print(f"❓ 问：{q}")
    # Qwen 首轮对话官方推荐 history=None
    response, _ = model.chat(tokenizer, q, history=None)
    print(f"💡 答：\n{response}")
    print("-" * 60)

1. 正在加载 Tokenizer...
2. 正在加载 Qwen-7B 模型 (14GB体积，绝对不会爆内存)...


Loading checkpoint shards: 100%|██████████| 8/8 [01:04<00:00,  8.07s/it]



✅ Qwen-7B-Chat 加载成功！内存安全，开始极速回答作业问题：

❓ 问：请说出以下两句话区别在哪里？ 1、冬天：能穿多少穿多少 2、夏天：能穿多少穿多少
💡 答：
这两句话的意思是相同的，都表达了“尽可能多地穿上衣服”的意思。但是它们的语境和时间不同。第一句话是在描述冬天的情况，强调在寒冷的天气里要保暖；而第二句话是在描述夏天的情况，强调在炎热的天气里要防晒。因此，尽管这两句话的意思相同，但表达的情感和场景却有所不同。
------------------------------------------------------------
❓ 问：请说出以下两句话区别在哪里？单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上
💡 答：
这两句话的区别在于它们描述了单身狗产生原因的不同方面。第一句话指出单身狗产生原因是“谁都看不上”，意味着他们在择偶市场上缺乏吸引力或者不符合他人的标准；而第二句话则是将这种原因拆分成两个部分：“一是谁都看不上”和“二是谁都看不上”，这意味着单身狗在个人魅力和自身条件上都存在问题。因此，这两句话从不同的角度解释了单身狗产生的原因，并强调了不同的因素可能对一个人的恋爱生活造成影响。
------------------------------------------------------------
❓ 问：他知道我知道你知道他不知道吗？ 这句话里，到底谁不知道
💡 答：
这句话的含义是：这个人知道你知道我知道他知道他不知道这个事实吗？

在这个语句中，我们知道以下几点：

1. 第一个人知道“我知道他知道他知道他不知道”这个事实。
2. 第二个人知道第一个人知道这个事实。
3. 第三个人知道第一个人知道第一个人知道这个事实。

因此，第一个人知道第二个人和第三个人都不知道他不知道这个事实。
------------------------------------------------------------
❓ 问：明明明明明白白白喜欢他，可她就是不说。 这句话里，明明和白白谁喜欢谁？
💡 答：
这句话里的“她”喜欢“他”。
------------------------------------------------------------
❓ 问：领导：你这是什么意思？ 小明：没什么意思。意思意思。 领导：你

In [1]:
import torch
import warnings
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedModel

# 🚀 补丁：解决新版本 transformers 导致的兼容性报错
if not hasattr(PreTrainedModel, "all_tied_weights_keys"):
    PreTrainedModel.all_tied_weights_keys = property(lambda self: {})

# 📁 设置本地路径
model_path = "/mnt/workspace/deepseek-llm-7b-chat"

print("1. 正在加载 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

print("2. 正在加载 DeepSeek-7B-Chat (使用 bfloat16，约占 14.5GB 内存)...")
# low_cpu_mem_usage=False 避开死锁 Bug
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    trust_remote_code=True, 
    torch_dtype=torch.bfloat16, 
    low_cpu_mem_usage=False
).eval() 

print("\n✅ DeepSeek 加载成功！开始回答测试问题：\n")
print("="*60)

# 📝 作业要求的 5 个中文语义测试题
questions = [
    "请说出以下两句话区别在哪里？ 1、冬天：能穿多少穿多少 2、夏天：能穿多少穿多少",
    "请说出以下两句话区别在哪里？单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上",
    "他知道我知道你知道他不知道吗？ 这句话里，到底谁不知道",
    "明明明明明白白白喜欢他，可她就是不说。 这句话里，明明和白白谁喜欢谁？",
    "领导：你这是什么意思？ 小明：没什么意思。意思意思。 请问：以上“意思意思”是什么意思。"
]

# DeepSeek 官方推荐的简易对话模版
def ask_deepseek(question):
    messages = [{"role": "user", "content": question}]
    # 构造 Prompt
    input_tensor = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    
    outputs = model.generate(input_tensor, max_new_tokens=300, do_sample=False)
    
    # 只显示回答部分
    result = tokenizer.decode(outputs[0][input_tensor.shape[1]:], skip_special_tokens=True)
    return result

for q in questions:
    print(f"❓ 问：{q}")
    try:
        response = ask_deepseek(q)
        print(f"💡 答：\n{response}")
    except Exception as e:
        print(f"❌ 运行出错: {e}")
    print("-" * 60)

1. 正在加载 Tokenizer...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


2. 正在加载 DeepSeek-7B-Chat (使用 bfloat16，约占 14.5GB 内存)...


Loading checkpoint shards: 100%|██████████| 2/2 [00:35<00:00, 17.58s/it]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.



✅ DeepSeek 加载成功！开始回答测试问题：

❓ 问：请说出以下两句话区别在哪里？ 1、冬天：能穿多少穿多少 2、夏天：能穿多少穿多少


...

💡 答：
这两句话在表达上有些相似，但含义和侧重点有所不同。

第一句话：“单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上。”这句话强调了单身狗产生的原因有两个，即“谁都看不上”和“谁都看不上”。这里的“谁都看不上”可以理解为单身狗在选择伴侣时对对方的标准较高，或者是对自己不够自信，不愿意轻易接受他人。而“谁都看不上”则强调了单身狗在选择伴侣时可能存在的一种心理状态，即对所有潜在的伴侣都不满意，没有找到合适的对象。

第二句话：“单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上。”这句话与第一句话在表达上有些相似，但侧重点和表达方式略有不同。这句话强调了单身狗产生的原因有两个，即“谁都看不上”和“谁都看不上”。这里的“谁都看不上”可以理解为单身狗在选择伴侣时对对方的标准较高，或者是对自己不够自信，不愿意轻易接受他人。而“谁都看不上”则强调了单身狗在选择伴侣时可能存在的一种心理状态，即对所有潜在的伴侣都不满意，没有找到合适的对象。

总的来说，两句话在表达上有些相似，但第一句话更侧重于强调单身狗产生的原因有两个，而第二句话则更
------------------------------------------------------------
❓ 问：他知道我知道你知道他不知道吗？ 这句话里，到底谁不知道


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


💡 答：
这句话的结构比较复杂，似乎是在描述一个循环推理的情况。如果我们将这句话简化，可以理解为：

- “他”知道“你知道”他不知道。
- “你知道”他不知道。
- 所以，“他”不知道“你知道”他不知道。

因此，在这个循环推理中，最终可以得出“他”不知道“你知道”他不知道。
------------------------------------------------------------
❓ 问：明明明明明白白白喜欢他，可她就是不说。 这句话里，明明和白白谁喜欢谁？


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


💡 答：
这句话中，“明明明白白白喜欢他”，表示“明明”喜欢“他”，而“白白”对“他”的情感没有明确说明，但可以理解为“白白”对“他”有好感。所以，“明明”喜欢“他”，而“白白”对“他”有好感。
------------------------------------------------------------
❓ 问：领导：你这是什么意思？ 小明：没什么意思。意思意思。 请问：以上“意思意思”是什么意思。
💡 答：
"意思意思"是一种汉语表达方式，通常用来表示“稍微表示一下”或者“略表心意”。在小明的回答中，他可能是在说他的话没有特别的含义，只是想表达一下自己的态度或者立场。这是一种比较含蓄的表达方式，可能没有直接说出具体的意思，而是通过这种方式来传达自己的想法。
------------------------------------------------------------
